# 03｜把方案连起来：公共微调数据、原始计数、验证和预算

**目标：** 明确缺什么训练资产；生成一份 400 行的教学 H5AD；观察格式检查与性能评估的区别；用可调参数估算训练资源。

这份笔记可以独立运行，默认 CPU，产物只写 `output/notebook-learning/03-pipeline/`。运行全部单元不会下载大数据、启动正式微调或提交比赛。

前两课：[真实数据](01_vc2026_data.ipynb) → [模型与迁移](02_state_model_and_transfer.ipynb)；完整方案：[L3-04 State 微调教程](../docs/lessons/L3-04-State微调实战与算力预算.md)。

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display, Markdown

# 从仓库根目录或 notebook/ 启动都可以；不写死某台机器的绝对路径。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').exists()
             and (p / 'docs/Official-website').is_dir()), None)
if ROOT is None:
    raise RuntimeError('请在 virtual-cell-2026 仓库内启动 notebook')
DATA = ROOT / 'data/vcc2026-validation/controls'
if not (DATA / 'manifest.json').exists():
    raise FileNotFoundError(f'缺少 {DATA}；请先放好官方 controls 数据包')
OUT = ROOT / 'output/notebook-learning/03-pipeline'
OUT.mkdir(parents=True, exist_ok=True)
manifest = json.loads((DATA / 'manifest.json').read_text())
genes = pd.read_csv(DATA / 'gene_names.csv')['gene_name'].astype(str).tolist()
targets = pd.read_csv(DATA / 'pert_counts.csv')['target_gene'].astype(str).tolist()
contexts = manifest['contexts']
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'],
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.dpi': 110,
    'svg.fonttype': 'none', 'pdf.fonttype': 42,
})
COLORS = {'A': '#167B72', 'B': '#3E71AD', 'C': '#DC654F'}
pd.set_option('display.max_rows', 12)
pd.set_option('display.max_columns', 12)
def show_figure(fig, name):
    # 完整可再生成图只写入 Git 忽略目录；notebook 内嵌适合阅读的预览。
    fig.savefig(OUT / f'{name}.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUT / f'{name}.svg', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)
print('Python:', sys.version.split()[0], '| 数据目录:', DATA.relative_to(ROOT))
print('本课输出目录:', OUT.relative_to(ROOT))

Python: 3.13.14 | 数据目录: data/vcc2026-validation/controls
本课输出目录: output/notebook-learning/03-pipeline


## 单元一：把“需要数据”变成可检查清单

监督训练需要公共背景中**匹配的 NTC 与扰动后细胞**，以及靶点标签。2026 controls 只提供输入背景，不能从中直接提取真实扰动监督。

| 来源 | 学到什么 | 首轮角色 |
|---|---|---|
| Replogle K562 essential / RPE1 | 同一靶点在不同背景的响应 | 核心训练/留出 |
| Nadig HepG2 / Jurkat | 增加背景差异 | 核心训练/留出 |
| K562 GWPS | 增加有监督靶点覆盖 | 第二批加入 |
| 2025 H1 | 接近本赛 Flex 平台的另一个背景 | 先整体留出作开发评分 |
| ESM2 特征 | 编码蛋白靶基因的连续输入 | 确保当轮所有靶点可表示 |

下面文件大小与 MD5 来自 2026-09-14 核验的固定 scPerturb Zenodo 记录。这里只列清单，不自动下载；可在完整教程找到 curl 示例。文件名 `NadigOConner2024` 与 2025 正式论文是版本命名关系，不是两份独立实验。

In [2]:
training_files = pd.DataFrame([
    ('ReplogleWeissman2022_K562_essential.h5ad', 1546729675, 'core'),
    ('ReplogleWeissman2022_rpe1.h5ad', 1236886900, 'core'),
    ('NadigOConner2024_hepg2.h5ad', 850590740, 'core'),
    ('NadigOConner2024_jurkat.h5ad', 1293665804, 'core'),
    ('ReplogleWeissman2022_K562_gwps.h5ad', 8805466154, 'expand'),
], columns=['file','bytes','stage'])
PUBLIC = ROOT / 'data/state-training/raw/scperturb'  # 计划中的目录，可修改
training_files['download_GB'] = training_files['bytes'] / 1e9
training_files['local_exists'] = training_files['file'].map(lambda f: (PUBLIC/f).exists())
display(training_files.drop(columns='bytes'))
print('核心四文件 GB:', training_files.loc[training_files.stage.eq('core'),'bytes'].sum()/1e9)
print('加 GWPS 后 GB:', training_files['bytes'].sum()/1e9)
display(Markdown('[固定下载记录](https://zenodo.org/records/13350497)'))

,file,stage,download_GB,local_exists
0,ReplogleWeissman2022_K562_essential.h5ad,core,1.546730,False
1,ReplogleWeissman2022_rpe1.h5ad,core,1.236887,False
2,NadigOConner2024_hepg2.h5ad,core,0.850591,False
3,NadigOConner2024_jurkat.h5ad,core,1.293666,False
4,ReplogleWeissman2022_K562_gwps.h5ad,expand,8.805466,False


核心四文件 GB: 4.927873119
加 GWPS 后 GB: 13.733339273


[固定下载记录](https://zenodo.org/records/13350497)

### 训练数据的统一接口

每份训练数据保留 raw `.X`、统一 `target_gene/context_id/batch_id`、原 guide/study/assay，并明确实测基因 mask。**批次效应（batch effect）**是实验处理/测量导致的系统差异；把不同条件的对照配给扰动细胞，可能把技术差异学成扰动响应。

HVG 路线另存 `obsm['X_hvg']`、与之完全同序的 `uns['hvg_names']` 和 `var['gene_name']`；不允许旧权重的基因顺序被重新选择的 HVG 替代。缺测基因不能补零当作实测零；mask 需要进入损失，原生 State 不替我们自动完成。

一个集合来自同一 `context × target × batch`；NTC 来自同一文件/split 内的匹配背景与批次。共享细胞身份或 guide 的随机拆行不等于未知背景验证。

In [3]:
protocol = pd.DataFrame([
    ('public_train', '公共训练背景', True, '模型训练'),
    ('public_val', '留出的开发背景/目标', True, '选 checkpoint；记录是否已暴露父权重'),
    ('public_test', '独立留出的背景/目标', True, '只做冻结评估'),
    ('ABC_controls', '官方匿名背景 NTC', False, '推理条件，不当扰动标签'),
    ('H1_reference', 'H1 真值/DE/缩放锚点', True, '只给评分器；训练不可读'),
], columns=['split','content','has_perturbed_truth','allowed_use'])
display(protocol)

,split,content,has_perturbed_truth,allowed_use
0,public_train,公共训练背景,True,模型训练
1,public_val,留出的开发背景/目标,True,选 checkpoint；记录是否已暴露父权重
2,public_test,独立留出的背景/目标,True,只做冻结评估
3,ABC_controls,官方匿名背景 NTC,False,推理条件，不当扰动标签
4,H1_reference,H1 真值/DE/缩放锚点,True,只给评分器；训练不可读


**预训练暴露也计入划分。** 父 checkpoint 已见过某背景，之后只在微调 TOML 里删掉它，不能称为干净 LOCO（leave-one-context-out，整背景留出）。`zeroshot/jurkat` 有一致的官方留出协议线索，但缺完整实际训练 manifest；保持这一不确定性。

若 H1 被多次用于挑超参数，就称开发集。最终结构冻结后可以加入 H1 训练，但之后不再把 H1 分数称为未见背景结果。A/B/C 的标签不可见，不能凭空填写离线成绩。

## 单元二：看懂一条真正的微调命令

下一单元从已核对的 Markdown 教程提取原生 CLI 示例，并只改工作路径。它**展示并保存命令文本，不执行**。阅读时对照上一课的配置：328 主干、set64、`init_from`、ESM2、新 run 目录，以及原生/项目适配边界。

正式 State 环境需要 Python 3.11/3.12；当前教学环境是 3.13，二者分开。缺少权重、对齐训练数据和特征时，不能让“Run All”意外开始长时间训练。

In [4]:
import re
text = (ROOT / 'docs/lessons/L3-04-State微调实战与算力预算.md').read_text()
fence = chr(96)*3
blocks = re.findall('^'+fence+r'bash\n(.*?)^'+fence, text, flags=re.M|re.S)
command = next(block.strip() for block in blocks if block.startswith('state tx train'))
WORK = ROOT / 'data/state-training'
command = command.replace('/data/vcc2026', str(WORK))
display(Markdown(f'{fence}bash\n{command}\n{fence}'))
(OUT / 'planned_train_command.txt').write_text(command+'\n')
required = {
    'parent_checkpoint': WORK / 'pretrained/ST-HVG-Replogle/zeroshot/jurkat/checkpoints/best.ckpt',
    'aligned_split': WORK / 'splits/hvg_pilot.toml',
    'target_features': WORK / 'features/esm2_all_targets.pt',
}
display(pd.DataFrame([{'asset': name, 'exists': path.exists(), 'path': str(path.relative_to(ROOT))}
                      for name,path in required.items()]))
print('这里仅检查存在性；存在不等于通过内容、来源和语义校验。')

```bash
state tx train \
  model=state \
  model.kwargs.init_from=/home/caii/projects/virtual-cell-2026/data/state-training/pretrained/ST-HVG-Replogle/zeroshot/jurkat/checkpoints/best.ckpt \
  model.kwargs.hidden_dim=328 \
  model.kwargs.cell_set_len=64 \
  model.kwargs.transformer_backbone_kwargs.hidden_size=328 \
  model.kwargs.transformer_backbone_kwargs.intermediate_size=3072 \
  model.kwargs.transformer_backbone_kwargs.num_hidden_layers=8 \
  model.kwargs.transformer_backbone_kwargs.num_attention_heads=12 \
  model.kwargs.transformer_backbone_kwargs.num_key_value_heads=12 \
  model.kwargs.transformer_backbone_kwargs.head_dim=64 \
  model.kwargs.n_encoder_layers=1 \
  model.kwargs.n_decoder_layers=1 \
  model.kwargs.batch_encoder=false \
  model.kwargs.use_batch_token=false \
  model.kwargs.freeze_pert_backbone=false \
  model.kwargs.lora.enable=false \
  +model.kwargs.gene_decoder_bool=false \
  model.kwargs.log1p_from_raw_counts=false \
  data.kwargs.toml_config_path=/home/caii/projects/virtual-cell-2026/data/state-training/splits/hvg_pilot.toml \
  data.kwargs.embed_key=X_hvg \
  data.kwargs.output_space=gene \
  data.kwargs.pert_col=target_gene \
  data.kwargs.cell_type_key=context_id \
  data.kwargs.batch_col=batch_id \
  data.kwargs.basal_mapping_strategy=batch \
  data.kwargs.control_pert=non-targeting \
  data.kwargs.perturbation_features_file=/home/caii/projects/virtual-cell-2026/data/state-training/features/esm2_all_targets.pt \
  data.kwargs.num_workers=4 \
  +data.kwargs.is_log1p=false \
  training.batch_size=4 \
  training.gradient_accumulation_steps=4 \
  training.lr=0.00001 \
  training.max_steps=2000 \
  training.val_freq=2000 \
  training.train_seed=42 \
  training.devices=1 \
  use_wandb=false \
  training.wandb_track=false \
  output_dir=/home/caii/projects/virtual-cell-2026/data/state-training/runs \
  name=st_hvg_esm2_pilot_001
```

,asset,exists,path
0,parent_checkpoint,False,data/state-training/pretrained/ST-HVG-Replogle...
1,aligned_split,False,data/state-training/splits/hvg_pilot.toml
2,target_features,False,data/state-training/features/esm2_all_targets.pt


这里仅检查存在性；存在不等于通过内容、来源和语义校验。


值得逐项检查的执行语义：

- `init_from` 按当前结构加载同名同形参数，开始新的优化过程；已有 `last.ckpt` 会优先 resume。
- 换基因轴/特征语义时，必须按名称迁移或显式重建；`strict=False` 不能解决生物坐标错位。
- `freeze_pert_backbone` 还会冻结 `project_out`；正式新头热身需显式设可训练范围。
- 固定源码默认没有为 ST 开启 bf16，分组学习率也需实现。
- `val_freq` 用于验证时按训练 batch，用于保存时却按 optimizer step；`ckpt_every_n_steps` 未被使用。正式选模前修正 callback，使权重与当次验证分数对应。

它们解释了为何我们先做 100–200 个更新的资源/接口检查，再做 2,000-step pilot，最后扩大训练。

## 单元三：从表达 rate 到一个 400 行的 H5AD

State 的 log 表达输出不能直接提交，必须接**全基因 rate 与文库量生成器**。本单元用实际 NTC 构造一个**无扰动效应的教学生成器**，只演示采样、整数约束和格式；它没有学会任何目标响应，不是 State 输出，不作正式提交。

我们从 A 的 46 个 NTC guide 平衡选取 400 个细胞（每 guide 8 或 9 个），以每细胞的相对表达为模板，向这 400 个模板的平均组成收缩 10%，再按原总量做多项式采样。10% 是教学设置，不是已验证的最佳值。

这种选择是为了模拟一次规定的 400-cell 输出，不能把本单元样本统计推广为整个背景的精确统计。上一课的全量 QC 才使用所有细胞。

In [5]:
CONTEXT = 'A'
DEMO_TARGET = 'ADNP'
N_OUTPUT = 400
SEED = 42
SHRINKAGE = 0.10
assert CONTEXT in contexts and DEMO_TARGET in targets
assert N_OUTPUT == manifest['cells_per_pert']
assert 0 <= SHRINKAGE <= 1
rng = np.random.default_rng(SEED)
a = ad.read_h5ad(DATA / f'context_{CONTEXT}.h5ad', backed='r')
try:
    ids = a.obs['ntc_id'].astype(str).to_numpy()
    guide_order = np.array(sorted(set(ids)))
    rng.shuffle(guide_order)  # 余数随机分配，避免总偏向排序靠前的 guide
    quotient, remainder = divmod(N_OUTPUT, len(guide_order))
    selected = np.sort(np.concatenate([
        rng.choice(np.flatnonzero(ids==guide), quotient+(i<remainder), replace=False)
        for i,guide in enumerate(guide_order)
    ]))
    raw_templates = a.X[selected].tocsr().astype(np.float64)
    source_cell_ids = a.obs_names[selected].tolist()
    guide_counts = pd.Series(ids[selected]).value_counts()
finally:
    a.file.close()
library_sizes = np.asarray(raw_templates.sum(axis=1)).ravel().astype(np.int64)
assert (library_sizes > 0).all() and (library_sizes <= 1_000_000).all()
assert len(selected) == len(set(selected)) == 400
print('选中细胞:', len(selected), '| 每 guide:', int(guide_counts.min()), '到', int(guide_counts.max()))
composition = raw_templates.multiply(1/library_sizes[:,None]).toarray()
mean_composition = composition.mean(axis=0, keepdims=True)
rate = (1-SHRINKAGE)*composition + SHRINKAGE*mean_composition
assert np.allclose(rate.sum(axis=1),1)
print('rate shape:', rate.shape, '| 当前没有应用任何 target 特异效应')

选中细胞: 400 | 每 guide: 8 到 9
rate shape: (400, 18533) | 当前没有应用任何 target 特异效应


**多项式采样（multinomial sampling）**把一个细胞的 L 个检测计数按基因概率分配，保证行和固定为 L。它不是一次新的生物实验，只是我们选定的观测模型。

限制：NTC 自带测量噪声，再采样可能叠加噪声；保持 NTC 总量忽略扰动改变总 RNA/捕获效率的可能性；没有响应模型就无法恢复下游差异。后续要在公共留出真值上比较均值、方差、零、协变和六指标，不能因为输出整数就宣布成功。

In [6]:
probability = rate / rate.sum(axis=1, keepdims=True)
rows = [sparse.csr_matrix(rng.multinomial(int(depth), p).astype(np.int32)[None,:])
        for p,depth in zip(probability,library_sizes)]
generated = sparse.vstack(rows, format='csr')
generated.eliminate_zeros()
assert np.array_equal(np.asarray(generated.sum(axis=1)).ravel(), library_sizes)
assert generated.shape == (400,18533)
display(pd.DataFrame([
    {'kind': '400 NTC templates', 'median_total': np.median(library_sizes),
     'median_detected_genes': np.median(raw_templates.getnnz(axis=1))},
    {'kind': 'teaching generation', 'median_total': np.median(np.asarray(generated.sum(axis=1))),
     'median_detected_genes': np.median(generated.getnnz(axis=1))},
]))
print('总量一致，但检测基因数等分布可以变化；这正是生成器要验证的地方。')

,kind,median_total,median_detected_genes
0,400 NTC templates,20644.0,6220.5
1,teaching generation,20644.0,5422.0


总量一致，但检测基因数等分布可以变化；这正是生成器要验证的地方。


### 写出、重新读回、再检查

`obs.target_gene=ADNP` 只是给教学输出标注希望模拟的条件，不会自动把 NTC 变成真正的 ADNP 扰动。我们用醒目的文件名和 `uns` 来源字段记录这一点。

文件仅含 A/ADNP 一个组合，能展示单片合同，但不满足完整 3×300 个组合。既不把 NTC 原始文件覆盖，也不调用 `vcc submit`。

In [7]:
prediction = ad.AnnData(
    X=generated,
    obs=pd.DataFrame({'context': CONTEXT, 'target_gene': DEMO_TARGET},
                     index=[f'TEACHING_{CONTEXT}_{DEMO_TARGET}_{i:04d}' for i in range(N_OUTPUT)]),
    var=pd.DataFrame(index=pd.Index(genes,name='gene_name')),
)
prediction.uns['purpose'] = 'TEACHING ONLY: no-effect NTC sampler; not State; not for submission'
prediction.uns['seed'] = SEED
demo_path = OUT / 'TEACHING_ONLY_not_for_submission.h5ad'
prediction.write_h5ad(demo_path,compression='gzip')
roundtrip = ad.read_h5ad(demo_path,backed='r')
try:
    print(roundtrip)
    assert roundtrip.var_names.tolist() == genes
    assert roundtrip.shape == (400,18533)
    assert (roundtrip.X[:].tocsr()!=generated).nnz == 0
    display(roundtrip.obs.head())
finally:
    roundtrip.file.close()
print('文件:', demo_path.relative_to(ROOT), '| MB:', round(demo_path.stat().st_size/1e6,2))
(OUT / 'demo_provenance.json').write_text(json.dumps({
    'purpose':'teaching only, no perturbation truth or State predictions',
    'context':CONTEXT,'target_label':DEMO_TARGET,'seed':SEED,'shrinkage':SHRINKAGE,
    'source_file':f'context_{CONTEXT}.h5ad','source_cell_ids':source_cell_ids,
},ensure_ascii=False,indent=2))

AnnData object with n_obs × n_vars = 400 × 18533 backed at '/home/caii/projects/virtual-cell-2026/output/notebook-learning/03-pipeline/TEACHING_ONLY_not_for_submission.h5ad'
    obs: 'context', 'target_gene'
    uns: 'purpose', 'seed'
    layers: None (.X)


,context,target_gene
TEACHING_A_ADNP_0000,A,ADNP
TEACHING_A_ADNP_0001,A,ADNP
TEACHING_A_ADNP_0002,A,ADNP
TEACHING_A_ADNP_0003,A,ADNP
TEACHING_A_ADNP_0004,A,ADNP


文件: output/notebook-learning/03-pipeline/TEACHING_ONLY_not_for_submission.h5ad | MB: 5.16


8616

In [8]:
checks = {
    'finite nonnegative integer values': bool(np.isfinite(generated.data).all()
        and (generated.data>=0).all() and np.equal(generated.data,np.floor(generated.data)).all()),
    'full ordered gene axis': prediction.var_names.tolist()==genes,
    'row totals <= 1,000,000': bool((np.asarray(generated.sum(axis=1))<=1_000_000).all()),
    'sparse and no explicit zeros': sparse.issparse(generated) and bool((generated.data!=0).all()),
    'this context-target has 400 rows': len(prediction.obs)==400,
    'all three contexts': set(prediction.obs.context)==set(contexts),
    'all 300 targets': set(prediction.obs.target_gene)==set(targets),
    'complete 360,000 rows': prediction.n_obs==len(contexts)*len(targets)*400,
}
display(pd.Series(checks,name='passes'))
print('后三项 False 是本课预期：这只是一个教学分片，不能提交。')

finite nonnegative integer values     True
full ordered gene axis                True
row totals <= 1,000,000               True
sparse and no explicit zeros          True
this context-target has 400 rows      True
all three contexts                   False
all 300 targets                      False
complete 360,000 rows                False
Name: passes, dtype: bool

后三项 False 是本课预期：这只是一个教学分片，不能提交。


**诊断题：** 为什么前五项通过，却仍不能说“完成了 VC2026”？

<details><summary>参考答案</summary>还缺全背景/靶点覆盖、学习得到的扰动响应、真实六指标评估与正式打包。甚至完整格式通过也只能证明合同，不能证明生物预测准确。</details>

## 单元四：验证与打包各检查哪一层

| 阶段 | 输入 | 能确认什么 | 不能确认什么 |
|---|---|---|---|
| 本笔记分片检查 | 400×18,533 教学 H5AD | 数值、形状、列顺序 | 真正扰动响应 |
| H1 开发评分 | 50,400×18,080；126 targets；H1 参考 | 同一个 H1 面板内的六指标变化 | ABC/DEF 的分数或干净预训练暴露 |
| 公共背景留出 | 未暴露背景 NTC + 隔离真值 | 跨背景迁移证据 | 所有未来背景均有效 |
| `vcc prep --dry-run` | 360,000×18,533；3×300×400 | 官方文件合同 | 生物质量、训练来源无泄漏 |
| 服务器评分 | 完整 `.vcc` | 官方当轮隐藏真值评分 | 下一轮必然达到相同分数 |

六指标包括 pds、mse、nmae、fid、reach、jac；fid 是响应方向 fidelity，不是图像任务的 Fréchet FID。raw counts 交给评分器后，评分器按各指标规则内部变换。不能先 log1p 文件后仍称 raw counts。

基因轴不同的 H1 与官方文件要分别导出。参考 DE、moments 和 anchors 都含真值信息；训练不能读取。实际接入命令见 [H1 审计](../docs/research/h1-benchmark-audit.md)。

## 单元五：在花 GPU 时间之前，先填资源表

**工程起点，尚非实测最低配置：** 单张 24 GB GPU、64 GB 主机 RAM、300 GB SSD，先预留 4–8 GPU 小时做加载和 pilot；扩到官方全基因路径可考虑 48 GB/128 GB/500 GB。当前学习笔记的 CPU 用量不能当作 State 训练吞吐。

训练总时间取决于**optimizer step** ，不是 microbatch 数。batch_size 是每个 microbatch 的集合数，梯度累积若为 A，则每更新约处理 `batch_size × A × GPU数` 个集合。集合会重复采样细胞，这个量不代表独立实验数。

下面是计算器而非报价/性能预测。先从真实 pilot 填 seconds/update 和验证耗时，再算总量。使用独立验证开销和 30% 余量，避免把所有开销藏进单步时间。

In [9]:
def budget_table(steps=10000, seconds_per_update=1.0, runs=6,
                 validation_minutes=5.0, validation_every_updates=1000,
                 gpu_count=1, price_per_gpu_hour=10.0, reserve=0.30):
    validation_events = steps // validation_every_updates
    training_hours = steps*seconds_per_update/3600
    validation_hours = validation_events*validation_minutes/60
    wall_per_run = (training_hours+validation_hours)*(1+reserve)
    gpu_hours = wall_per_run*runs*gpu_count
    return pd.DataFrame([{
        'training_h/run': training_hours, 'validation_h/run': validation_hours,
        'reserved_wall_h/run': wall_per_run, 'total_GPU_h': gpu_hours,
        'hypothetical_GPU_cost_CNY': gpu_hours*price_per_gpu_hour,
    }]).round(2)

display(budget_table())
microbatch, accumulation, set_length, gpu_count = 4,4,64,1
print('每更新集合数:', microbatch*accumulation*gpu_count)
print('每更新细胞位置:', microbatch*accumulation*gpu_count*set_length)
print('金额不含 CPU/RAM、磁盘、流量；单价是可修改的算术假设。')

,training_h/run,validation_h/run,reserved_wall_h/run,total_GPU_h,hypothetical_GPU_cost_CNY
0,2.78,0.83,4.69,28.17,281.67


每更新集合数: 16
每更新细胞位置: 1024
金额不含 CPU/RAM、磁盘、流量；单价是可修改的算术假设。


In [10]:
import ipywidgets as widgets
# Jupyter/VS Code 支持 widgets 时可拖动；不支持时直接修改上一个代码格的参数。
def show_budget(seconds_per_update, steps, runs, price_per_gpu_hour):
    display(budget_table(steps=steps,seconds_per_update=seconds_per_update,
                         runs=runs,price_per_gpu_hour=price_per_gpu_hour))
budget_widget = widgets.interactive(
    show_budget,
    seconds_per_update=widgets.FloatSlider(value=1,min=0.25,max=10,step=0.25,description='秒/更新'),
    steps=widgets.IntSlider(value=10000,min=2000,max=40000,step=2000,description='更新次数'),
    runs=widgets.IntSlider(value=6,min=1,max=12,step=1,description='实验个数'),
    price_per_gpu_hour=widgets.FloatSlider(value=10,min=1,max=50,step=1,description='元/GPU时'),
)
display(budget_widget)

interactive(children=(FloatSlider(value=1.0, description='秒/更新', max=10.0, min=0.25, step=0.25), IntSlider(val…

### 为什么磁盘文件很小，内存仍可能爆掉

H5AD 可以压缩，读出来的数组却不按压缩文件体积占内存。以下只做算术，不分配完整输出。CSR 索引类型会影响成本，库可能将它升级到 int64；显式零也计入 stored items。

全参数 FP32 Adam 的粗略参数/梯度/状态项约 `16 × 参数数 bytes`，还没有激活、距离矩阵和 CUDA 缓冲。LoRA/冻结可以减少部分梯度与状态，但不是把所有显存自动除以同一个倍率。

In [11]:
n_rows, n_genes = len(contexts)*len(targets)*400, len(genes)
assumed_nnz_per_cell = 5800  # 容量假设，不是本次生成器的实测承诺
nnz = n_rows*assumed_nnz_per_cell
display(pd.DataFrame([
    {'layout':'one dense float32 matrix','GiB':n_rows*n_genes*4/2**30},
    {'layout':'two dense float32 copies','GiB':n_rows*n_genes*8/2**30},
    {'layout':'CSR: int32 values/indices, int64 indptr','GiB':(nnz*8+(n_rows+1)*8)/2**30},
    {'layout':'CSR: int32 values, int64 indices/indptr','GiB':(nnz*12+(n_rows+1)*8)/2**30},
]).round(2))
print('本次教学输出实测 nnz/cell:', generated.nnz / generated.shape[0])
print('必须分片推理、即时写盘；全量打包还需单独测峰值。')

,layout,GiB
0,one dense float32 matrix,24.85
1,two dense float32 copies,49.71
2,"CSR: int32 values/indices, int64 indptr",15.56
3,"CSR: int32 values, int64 indices/indptr",23.34


本次教学输出实测 nnz/cell: 5259.0725
必须分片推理、即时写盘；全量打包还需单独测峰值。


## 完成本套学习后，可以自己检查的六件事

1. 我知道 A/B/C 文件的每一行是什么、为什么没有扰动真值。
2. 我能区分表达轴中的基因与输入条件中的靶点，以及 raw 值与存储 dtype。
3. 我能解释 State 的 B/S/G/H 含义，为什么集合监督不依赖细胞顺序。
4. 我知道微调要核对父权重、基因/靶点/batch 语义和预训练暴露。
5. 我能写出、读回并检查一个计数 H5AD，知道格式正确不等于预测好。
6. 我能用实际 step 时间、验证耗时和峰值内存更新采购预算。

真正实施时仍要完成：公共数据 ETL、特征覆盖、按名称迁移、缺测 mask、训练/保存修正、计数适配、离线评分、完整轮容量预演。这里没有把这些未实现组件伪装成已完成的训练结果。

来源：[详细方案](../docs/lessons/L3-04-State微调实战与算力预算.md)、[检查点源码审计](../docs/research/state-checkpoint-finetuning-audit.md)、[官方数据合同](../docs/Official-website/About-the-Data.md)、[当前规则与截止时间](../docs/research/submission-contract-check.md)。研究/资产来源与固定版本见这些文档及 `assets/state/sources.json`。